In [1]:
%%capture
!pip install gradio
!pip install transformers
!pip install torch

##**Deepset - Roberta-base-squad2**

This is the roberta-base model, fine-tuned using the SQuAD2.0 dataset. It's been trained on question-answer pairs, including unanswerable questions, for the task of Extractive Question Answering.

In [2]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

##Load the model, tockenizer, and pipeline from hugginface

In [3]:
tokenizer = AutoTokenizer.from_pretrained("deepset/roberta-base-squad2")
model = AutoModelForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

In [4]:
#Knowledge Base

context = """
Our return policy allows customers to return items within 30 days of purchase with a valid receipt.
The product price is $199.99 with a 10% discount currently available.
The available colors are red, blue, black, and white.
"""

In [8]:
def answer_customer_question(question):
    """
    Uses a Question Answering (QA) model to extract the most relevant answer
    from a given context based on the user's question.

    Args:
        question (str): The customer's question as input text.

    Returns:
        str: The model's predicted answer, or a fallback message if uncertain.
    """

    # Tokenize the input question along with the context text.
    # The tokenizer prepares the data for the model by encoding both question
    # and context into numerical input IDs, along with attention masks.
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

    # Disable gradient computation since this is inference (no training).
    # This reduces memory usage and speeds up execution.
    with torch.no_grad():
        outputs = model(**inputs)

    # Find the most probable start and end positions of the answer
    # within the context based on the model's predictions.
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1  # +1 to include the end token

    # Convert the token IDs in the predicted span back to readable text.
    # The model outputs indices corresponding to tokens; these are decoded into a string.
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs.input_ids[0][start:end])
    )

    return answer if answer.strip() else "I'm not sure. Please contact human support."


##Gradio, Simple Chatbot Interface for our Customer Service

**Gradio** is a Python library that allows you to quickly create easy-to-use **web interfaces** for machine learning models or Python functions. It’s commonly used to demo, test, or share models interactively without writing complex frontend code. In your example, `gr.Interface()` wraps the `answer_customer_question` function in a simple web app where users can type a question (`inputs="text"`) and see the model’s response (`outputs="text"`). The `title` sets the interface name (“Customer Support Chatbot”), and `iface.launch()` starts a local web server, opening the interface in a browser so anyone can interact with your chatbot in real time.


In [9]:
iface = gr.Interface(
    fn=answer_customer_question,
    inputs="text",
    outputs="text",
    title="Customer Support Chatbot"
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a3168ba9b7b6056de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
